# 00 – Bee Haven: Steuer-Notebook Silver (Stufe `silver_automatisierung`)

**Ziel:** Die Silver-Stufe mit **einem** Start ausführen: je Stadt **N1 Bereinigung → N2 Qualität → N3 Wetter** nacheinander.

**Warum ein Steuer-Notebook?** (Pflichtenheft PH-ITS-002, Karte 4.1, Entscheidung 30.09.2026, Konzeptgedanke K6 c)
- In Azure startet jede Notebook-Aktivität eine **eigene Spark-Sitzung** (Minuten Startzeit, Kosten). Das Steuer-Notebook startet die Teil-Notebooks mit `mssparkutils.notebook.runMultiple()` in **einer** Sitzung.
- Die Notebooks bleiben **einzeln und übersichtlich**; für jedes Teil-Notebook gibt es in Synapse einen eigenen Ausführungs-Schnappschuss (Fehlersuche).
- Vorbild: Kurs-Notebook 04 (Automatisierung, Schleife über alle Städte).

**Einfach erklärt:** Statt für jedes Notebook einen Rechner hochzufahren, fährt das Steuer-Notebook einmal hoch und arbeitet die Liste ab – Stadt für Stadt, Schritt für Schritt.

**Zwei Betriebsarten (automatisch erkannt):**
- **Azure Synapse:** ruft die Synapse-Kopien `01_bereinigung_synapse`, `02_qualitaet_synapse`, `03_wetter_synapse` auf.
- **Lokal (VS Code):** führt die lokalen Notebooks N1–N3 nacheinander aus (Test ohne Azure, keine Kosten). Die gespeicherten Notebooks werden dabei nicht verändert.

## 0. Parameter

In [1]:
# --- Parameter (Parameters) – in Synapse als Parameterzelle markiert; die Pipeline kann Werte überschreiben ---
STAEDTE = ""                              # leer = alle Städte aus konfiguration/standorte.csv; sonst z. B. "schwartau" oder "schwartau,wurzburg"
STORAGE_ACCOUNT = "mabeehavenlakehouse"   # Speicherkonto (Storage Account, ADLS Gen2) des Lakehouse
CONTAINER = "lakehouse"                   # Container mit bronze/, silver/, gold/, konfiguration/
ZEITLIMIT_S = 3600                        # höchstens 1 Stunde für die ganze Stufe (Schutz vor endlosen, teuren Läufen)

## 1. Betriebsart erkennen und Städte bestimmen

In [2]:
# --- Bibliotheken laden ---
import time                               # Laufzeit messen
import pandas as pd                       # Standort-Tabelle lesen

# --- Betriebsart: Synapse (notebookutils vorhanden) oder lokal ---
try:
    from notebookutils import mssparkutils   # nur in Azure Synapse vorhanden
    AZURE = True
except ImportError:
    AZURE = False
print("Betriebsart:", "Azure Synapse" if AZURE else "lokal")

# --- Standort-Tabelle lesen (Azure: aus dem Lake; lokal: Spiegel im Projektordner) ---
if AZURE:
    tabelle = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/konfiguration/standorte.csv"
else:
    from pathlib import Path
    tabelle = Path.cwd().parent / "02_daten_ITS-Data" / "lakehouse" / "konfiguration" / "standorte.csv"
standorte = pd.read_csv(tabelle)
alle = standorte["stadt"].tolist()                                       # alle bekannten Städte
staedte = [s.strip() for s in STAEDTE.split(",") if s.strip()] or alle  # Parameter leer -> alle
unbekannt = [s for s in staedte if s not in alle]
if unbekannt:                                                            # unbekannte Stadt -> abbrechen statt falsch rechnen
    raise ValueError(f"Stadt fehlt in konfiguration/standorte.csv: {unbekannt}")
print("Städte:", staedte)

Betriebsart: lokal
Städte: ['schwartau']


**Ergebnis:** Betriebsart und Städte stehen fest.

## 2. Ablaufplan: je Stadt N1 → N2 → N3

Jede Stadt bekommt eine Kette aus drei Schritten; ein Schritt startet erst, wenn der vorige **erfolgreich** war. Städte laufen nacheinander (`concurrency` 1) – kleiner Spark-Pool, vorhersehbare Kosten.

In [3]:
# --- Ablaufplan (DAG = gerichteter Ablaufgraph, Format von mssparkutils.notebook.runMultiple) ---
SCHRITTE = [("n1", "01_bereinigung"), ("n2", "02_qualitaet"), ("n3", "03_wetter")]   # Kurzname, Notebook
aktivitaeten = []
for stadt in staedte:
    vorher = None
    for kurz, notebook in SCHRITTE:
        aktivitaeten.append({
            "name": f"{kurz}_{stadt}",                                    # eindeutiger Name je Schritt und Stadt
            "path": f"{notebook}_synapse" if AZURE else notebook,         # Synapse-Kopie bzw. lokales Notebook
            "timeoutPerCellInSeconds": 1800,                              # eine Zelle höchstens 30 Minuten
            "args": {"STADT": stadt},                                     # überschreibt die Parameterzelle
            "dependencies": [vorher] if vorher else [],                   # erst nach dem vorigen Schritt
        })
        vorher = f"{kurz}_{stadt}"
ablauf = {"activities": aktivitaeten, "timeoutInSeconds": ZEITLIMIT_S, "concurrency": 1}
for a in aktivitaeten:
    print(f"{a['name']:16} {a['path']:26} nach: {a['dependencies'] or '-'}")

n1_schwartau     01_bereinigung             nach: -
n2_schwartau     02_qualitaet               nach: ['n1_schwartau']
n3_schwartau     03_wetter                  nach: ['n2_schwartau']


**Ergebnis:** Ablaufplan mit 3 Schritten je Stadt.

## 3. Ausführen

**Azure:** `runMultiple` führt den Ablaufplan in dieser Spark-Sitzung aus; bei einem Fehler bricht die Stufe ab, die Pipeline meldet „Fehlgeschlagen“, die Rohdaten bleiben in `bronze/new` (Archiv läuft erst am Ende).
**Lokal:** Die Notebooks werden nacheinander mit `nbclient` ausgeführt; der Parameter `STADT` wird im Speicher gesetzt, die Dateien bleiben unverändert.

In [4]:
def lokal_ausfuehren(notebook, stadt):
    """Lokales Notebook im Speicher ausführen; Zeile STADT = "..." der ersten Codezelle durch die gewünschte Stadt ersetzen."""
    import re, nbformat
    from nbclient import NotebookClient
    nb = nbformat.read(Path.cwd() / f"{notebook}.ipynb", as_version=4)
    zelle = next(z for z in nb.cells if z.cell_type == "code" and re.search(r'^STADT = "', z.source, re.M))
    zelle.source = re.sub(r'^STADT = ".*?"', f'STADT = "{stadt}"', zelle.source, count=1, flags=re.M)
    NotebookClient(nb, timeout=1800, kernel_name="python3", resources={"metadata": {"path": str(Path.cwd())}}).execute()

start = time.time()
if AZURE:
    ergebnis = mssparkutils.notebook.runMultiple(ablauf)                # alles in EINER Spark-Sitzung
    print(ergebnis)
else:
    for a in aktivitaeten:                                               # Reihenfolge = Abhängigkeiten (je Stadt N1 -> N2 -> N3)
        t0 = time.time()
        lokal_ausfuehren(a["path"], a["args"]["STADT"])                  # Fehler -> Ausnahme -> Abbruch (wie in Azure)
        print(f"{a['name']:16} erfolgreich  {time.time() - t0:5.0f} s")
print(f"Stufe Silver fertig in {time.time() - start:.0f} s")

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


n1_schwartau     erfolgreich      3 s


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


n2_schwartau     erfolgreich      7 s


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


n3_schwartau     erfolgreich     84 s
Stufe Silver fertig in 94 s


**Ergebnis:** siehe Ausgabe – Laufzeit je Schritt.